In [4]:
pip install pandas numpy statsmodels plotly kaleido


   ---------------------------------------- 0.0/9.6 MB ? eta -:--:--
   ---- ----------------------------------- 1.0/9.6 MB 6.3 MB/s eta 0:00:02
   --------- ------------------------------ 2.4/9.6 MB 6.3 MB/s eta 0:00:02
   --------------- ------------------------ 3.7/9.6 MB 6.3 MB/s eta 0:00:01
   -------------------- ------------------- 5.0/9.6 MB 6.3 MB/s eta 0:00:01
   --------------------------- ------------ 6.6/9.6 MB 6.3 MB/s eta 0:00:01
   -------------------------------- ------- 7.9/9.6 MB 6.3 MB/s eta 0:00:01
   ------------------------------------- -- 8.9/9.6 MB 6.3 MB/s eta 0:00:01
   ---------------------------------------- 9.6/9.6 MB 6.2 MB/s  0:00:01
   ---------------------------------------- 0.0/37.1 MB ? eta -:--:--
   - -------------------------------------- 1.3/37.1 MB 6.3 MB/s eta 0:00:06
   -- ------------------------------------- 2.6/37.1 MB 6.3 MB/s eta 0:00:06
   ---- ----------------------------------- 3.9/37.1 MB 6.3 MB/s eta 0:00:06
   ----- ----------------


[notice] A new release of pip is available: 25.3 -> 26.0
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import plotly.graph_objects as go
import os

# --------------------------------------------------
# 1. Load data
# --------------------------------------------------
csv_path = r"C:\Users\TUF\Desktop\Architecture\UCL\Wearbledevice\datacsvrecordings\Final_Valence_004.csv"
df = pd.read_csv(csv_path)

features = ["Void", "Bio", "Tech", "Entropy"]
y = df["GSR"]
X = df[features]

# Add intercept
X = sm.add_constant(X)

# --------------------------------------------------
# 2. Run regression
# --------------------------------------------------
model = sm.OLS(y, X).fit()

coefficients = model.params[1:]   # exclude intercept
p_values = model.pvalues[1:]      # exclude intercept

# --------------------------------------------------
# 3. Radar chart — Significance (P-values)
# --------------------------------------------------
radar_fig = go.Figure()

radar_fig.add_trace(
    go.Scatterpolar(
        r=p_values.values,
        theta=features,
        fill='toself',
        name='P-value',
        marker=dict(size=8)
    )
)

radar_fig.update_layout(
    title="Significance (P-values) — GSR as Dependent Variable",
    template="plotly_dark",
    polar=dict(
        radialaxis=dict(
            visible=True,
            range=[0, 1],
            tickvals=[0.01, 0.05, 0.1, 0.5, 1],
            ticktext=["0.01", "0.05", "0.1", "0.5", "1"]
        )
    ),
    showlegend=False,
    width=700,
    height=700
)

# --------------------------------------------------
# 4. Bar chart — Regression Coefficients (β)
# --------------------------------------------------
bar_fig = go.Figure()

bar_fig.add_trace(
    go.Bar(
        x=coefficients.values,
        y=features,
        orientation='h',
        marker=dict(color=coefficients.values, colorscale='RdBu'),
        text=np.round(coefficients.values, 3),
        textposition='outside'
    )
)

bar_fig.update_layout(
    title="Regression Coefficients (β) — Effect on GSR",
    template="plotly_dark",
    xaxis_title="β value",
    yaxis_title="Street Elements",
    width=900,
    height=500,
    margin=dict(l=120, r=40, t=80, b=60)
)

# --------------------------------------------------
# 5. Export SVGs
# --------------------------------------------------
output_dir = os.path.dirname(csv_path)

radar_fig.write_image(
    os.path.join(output_dir, "GSR_pvalue_radar.svg"),
    format="svg",
    width=700,
    height=700
)

bar_fig.write_image(
    os.path.join(output_dir, "GSR_beta_coefficients.svg"),
    format="svg",
    width=900,
    height=500
)


In [2]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import plotly.graph_objects as go
import os

# --------------------------------------------------
# 1. Load and combine multiple CSV files
# --------------------------------------------------
csv_files = [
    r"C:\Users\TUF\Desktop\Architecture\UCL\Wearbledevice\datacsvrecordings\Final_001_with_sentiment.csv",
    r"C:\Users\TUF\Desktop\Architecture\UCL\Wearbledevice\datacsvrecordings\Final_Valence_002.csv",
    r"C:\Users\TUF\Desktop\Architecture\UCL\Wearbledevice\datacsvrecordings\Final_Valence_004.csv",
    r"C:\Users\TUF\Desktop\Architecture\UCL\Wearbledevice\datacsvrecordings\Walk3cleaned.csv"
]

# Read and concatenate
df_list = [pd.read_csv(path) for path in csv_files]
df = pd.concat(df_list, ignore_index=True)

# Optional: drop rows with missing values in required columns
features = ["Void", "Bio", "Tech", "Entropy"]
required_cols = features + ["GSR"]
df = df.dropna(subset=required_cols)

# --------------------------------------------------
# 2. Prepare regression variables
# --------------------------------------------------
y = df["GSR"]
X = df[features]

# Add intercept
X = sm.add_constant(X)

# --------------------------------------------------
# 3. Run regression
# --------------------------------------------------
model = sm.OLS(y, X).fit()

coefficients = model.params[1:]   # exclude intercept
p_values = model.pvalues[1:]      # exclude intercept

# --------------------------------------------------
# 4. Radar chart — Significance (P-values)
# --------------------------------------------------
radar_fig = go.Figure()

radar_fig.add_trace(
    go.Scatterpolar(
        r=p_values.values,
        theta=features,
        fill='toself',
        name='P-value',
        marker=dict(size=8)
    )
)

radar_fig.update_layout(
    title="Significance (P-values) — Combined Walks (GSR)",
    template="plotly_dark",
    polar=dict(
        radialaxis=dict(
            visible=True,
            range=[0, 1],
            tickvals=[0.01, 0.05, 0.1, 0.5, 1],
            ticktext=["0.01", "0.05", "0.1", "0.5", "1"]
        )
    ),
    showlegend=False,
    width=700,
    height=700
)

# --------------------------------------------------
# 5. Bar chart — Regression Coefficients (β)
# --------------------------------------------------
bar_fig = go.Figure()

bar_fig.add_trace(
    go.Bar(
        x=coefficients.values,
        y=features,
        orientation='h',
        marker=dict(
            color=coefficients.values,
            colorscale='RdBu',
            showscale=True
        ),
        text=np.round(coefficients.values, 3),
        textposition='outside'
    )
)

bar_fig.update_layout(
    title="Regression Coefficients (β) — Effect on GSR (Combined Dataset)",
    template="plotly_dark",
    xaxis_title="β value",
    yaxis_title="Street Elements",
    width=900,
    height=500,
    margin=dict(l=120, r=40, t=80, b=60)
)

# --------------------------------------------------
# 6. Export SVGs
# --------------------------------------------------
output_dir = os.path.dirname(csv_files[0])

radar_fig.write_image(
    os.path.join(output_dir, "GSR_pvalue_radar_combined.svg"),
    format="svg",
    width=700,
    height=700
)

bar_fig.write_image(
    os.path.join(output_dir, "GSR_beta_coefficients_combined.svg"),
    format="svg",
    width=900,
    height=500
)

# --------------------------------------------------
# 7. Optional: Print regression summary
# --------------------------------------------------
print(model.summary())


                            OLS Regression Results                            
Dep. Variable:                    GSR   R-squared:                       0.114
Model:                            OLS   Adj. R-squared:                  0.110
Method:                 Least Squares   F-statistic:                     32.11
Date:                Sun, 01 Feb 2026   Prob (F-statistic):           3.29e-25
Time:                        11:05:29   Log-Likelihood:                -5785.4
No. Observations:                1007   AIC:                         1.158e+04
Df Residuals:                    1002   BIC:                         1.161e+04
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const        429.4672    437.387      0.982      0.3

In [6]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import plotly.graph_objects as go
import os
from scipy import stats

# 1. Load
csv_files = [
    r"C:\Users\TUF\Desktop\Architecture\UCL\Wearbledevice\datacsvrecordings\Final_001_with_sentiment.csv",
    r"C:\Users\TUF\Desktop\Architecture\UCL\Wearbledevice\datacsvrecordings\Final_Valence_002.csv",
    r"C:\Users\TUF\Desktop\Architecture\UCL\Wearbledevice\datacsvrecordings\Final_Valence_004.csv",
    r"C:\Users\TUF\Desktop\Architecture\UCL\Wearbledevice\datacsvrecordings\Walk3cleaned.csv"
]

df_list = [pd.read_csv(f) for f in csv_files if os.path.exists(f)]
df = pd.concat(df_list, ignore_index=True)

features = ["Void", "Bio", "Tech", "Entropy"]
required_cols = features + ["GSR"]

# Ensure data is numeric (strips strings if any)
for col in required_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

df = df.dropna(subset=required_cols)
print(f"Post-NA rows: {len(df)}")

# 2. Smoothing - Fixed with min_periods=1
# This ensures we don't create new NaNs that dropna() would delete
df['GSR'] = df['GSR'].rolling(window=5, center=True, min_periods=1).mean()
print(f"Post-smoothing rows: {len(df)}")

# 3. Outlier Removal - Using a wider net
z_scores = np.abs((df[required_cols] - df[required_cols].mean()) / (df[required_cols].std() + 1e-9))
df = df[(z_scores < 5).all(axis=1)] 
print(f"Post-outlier rows: {len(df)}")

# 4. Regression
if len(df) > 5:
    # Scale
    for col in required_cols:
        df[col] = (df[col] - df[col].mean()) / (df[col].std() + 1e-9)

    y = df["GSR"]
    X = sm.add_constant(df[features])
    model = sm.OLS(y, X).fit()

    # Visualization
    bar_fig = go.Figure(go.Bar(
        x=model.params[1:],
        y=features,
        orientation='h',
        marker=dict(color=model.params[1:], colorscale='RdBu', showscale=True),
        text=np.round(model.params[1:], 3),
        textposition='outside'
    ))
    bar_fig.update_layout(title="Standardized & Smoothed GSR Analysis", template="plotly_dark")
    bar_fig.show()
    
    # Save
    output_path = os.path.join(os.path.dirname(csv_files[0]), "GSR_final_analysis.svg")
    bar_fig.write_image(output_path)
    print(model.summary())
else:
    print("Still no data. Check if your CSV columns are actually named 'GSR', 'Void', etc.")

Post-NA rows: 1007
Post-smoothing rows: 1007
Post-outlier rows: 1007


ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

In [14]:
import pandas as pd
import plotly.graph_objects as go
import os

# --------------------------------------------------
# 1. Load CSV
# --------------------------------------------------
csv_path = r"C:\Users\TUF\Desktop\Architecture\UCL\Wearbledevice\datacsvrecordings\Final_001_with_sentiment.csv"
df = pd.read_csv(csv_path)

features = ["Void", "Bio", "Tech", "Entropy"]

# --------------------------------------------------
# 2. Get highest & lowest GSR rows
# --------------------------------------------------
highest_gsr = df.nlargest(15, "GSR")
lowest_gsr = df.nsmallest(15, "GSR")

# --------------------------------------------------
# 3. Radar plot function
# --------------------------------------------------
def create_radar(data, title, filename):
    fig = go.Figure()

    for i, row in data.iterrows():
        fig.add_trace(
            go.Scatterpolar(
                r=row[features].values.tolist() + [row[features].values[0]],
                theta=features + [features[0]],
                fill='toself',
                name=f"GSR = {round(row['GSR'], 2)}"
            )
        )

    fig.update_layout(
        title=title,
        template="plotly_dark",
        polar=dict(
            radialaxis=dict(
                visible=True,
                showticklabels=True
            )
        ),
        width=700,
        height=700
    )

    fig.write_image(filename, format="svg", width=700, height=700)

# --------------------------------------------------
# 4. Export SVGs
# --------------------------------------------------
output_dir = os.path.dirname(csv_path)

create_radar(
    highest_gsr,
    "Top 15 Highest GSR Moments — Environmental & Sentiment Context",
    os.path.join(output_dir, "Highest_GSR_Radar.svg")
)

create_radar(
    lowest_gsr,
    "Top 15 Lowest GSR Moments — Environmental & Sentiment Context",
    os.path.join(output_dir, "Lowest_GSR_Radar.svg")
)

print("SVG radar graphs exported successfully.")


SVG radar graphs exported successfully.


In [16]:
import os
import shutil

# Source and destination folders
source_folder = r"C:\Users\TUF\Desktop\Architecture\UCL\Wearbledevice\datacsvrecordings\Pictures\Walk1"
dest_folder = r"C:\Users\TUF\Desktop\Architecture\UCL\Wearbledevice\datacsvrecordings\Pictures\Walk1_OnePerMinute"

os.makedirs(dest_folder, exist_ok=True)

seen_minutes = set()

for filename in sorted(os.listdir(source_folder)):
    if not filename.lower().endswith((".bmp", ".jpg", ".jpeg", ".png")):
        continue

    try:
        # Example: img_11-02-45_0.bmp
        time_part = filename.split("_")[1]  # 11-02-45
        hour, minute, _ = time_part.split("-")

        minute_key = f"{hour}-{minute}"

        if minute_key not in seen_minutes:
            seen_minutes.add(minute_key)
            src_path = os.path.join(source_folder, filename)
            dst_path = os.path.join(dest_folder, filename)
            shutil.copy(src_path, dst_path)

    except Exception as e:
        print(f"Skipped {filename}: {e}")

print(f"Done. Copied {len(seen_minutes)} images (one per minute).")


Done. Copied 33 images (one per minute).


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import wave
import contextlib

# Input and output paths
wav_path = r"C:\Users\TUF\Desktop\Architecture\UCL\Wearbledevice\datacsvrecordings\Music\reconstructed_DeviceSim_16k.wav"
svg_path = r"C:\Users\TUF\Desktop\Architecture\UCL\Wearbledevice\datacsvrecordings\Music\reconstructed_DeviceSim_16k.svg"

# Open WAV file
with contextlib.closing(wave.open(wav_path, 'r')) as wf:
    n_frames = wf.getnframes()
    framerate = wf.getframerate()
    audio_frames = wf.readframes(n_frames)
    n_channels = wf.getnchannels()
    sample_width = wf.getsampwidth()

# Convert audio bytes to numpy array
audio = np.frombuffer(audio_frames, dtype=np.int16)

# If stereo, take only first channel
if n_channels > 1:
    audio = audio[::n_channels]

# Create time axis in seconds
time = np.linspace(0, n_frames / framerate, num=n_frames)

# Stretch horizontally by factor of 2
plt.figure(figsize=(24, 4))  # doubled width
plt.plot(time, audio, color='blue')
plt.xlabel("Time (s)")
plt.ylabel("Amplitude")
plt.title("Waveform of reconstructed_DeviceSim_16k.wav")
plt.tight_layout()

# Save as SVG
plt.savefig(svg_path, format='svg')
plt.close()

print(f"SVG waveform saved to: {svg_path}")


In [6]:
import pandas as pd

# Input and output paths
input_csv = r"C:\Users\TUF\Desktop\Architecture\UCL\Wearbledevice\datacsvrecordings\Final_001_with_sentiment.csv"
output_csv = r"C:\Users\TUF\Desktop\Architecture\UCL\Wearbledevice\datacsvrecordings\GSR_Top15_Bottom15_Transcript.csv"

# Read CSV
df = pd.read_csv(input_csv)

# Ensure required columns exist
required_columns = ["Timestamp", "GSR", "Transcript"]
missing = [col for col in required_columns if col not in df.columns]
if missing:
    raise ValueError(f"Missing columns: {missing}")

# Drop rows with missing GSR or Transcript
df_clean = df.dropna(subset=["GSR", "Transcript"])

# Sort by GSR
df_sorted = df_clean.sort_values(by="GSR")

# Bottom 15 (lowest GSR)
low_15 = df_sorted.head(15).copy()
low_15["GSR_Category"] = "Low_GSR"

# Top 15 (highest GSR)
top_15 = df_sorted.tail(15).copy()
top_15["GSR_Category"] = "High_GSR"

# Combine results
result_df = pd.concat([low_15, top_15])

# Keep only required columns
result_df = result_df[["Timestamp", "GSR", "Transcript", "GSR_Category"]]

# Export to CSV
result_df.to_csv(output_csv, index=False)

print("Export completed successfully.")
print(f"File saved at:\n{output_csv}")


Export completed successfully.
File saved at:
C:\Users\TUF\Desktop\Architecture\UCL\Wearbledevice\datacsvrecordings\GSR_Top15_Bottom15_Transcript.csv
